In [1]:
import seqlib
import pandas as pd
import re

In [2]:
design = pd.read_csv("../Data/DesignFiles/DF_SOG1.csv")
design.drop_duplicates(subset="ArrayDNA", inplace=True)

In [3]:
def ADfinder(read):
    try: 
        #AD = re.split(r'GCGGGCTCTACTTCATCGGCTAGC', read)[1]
        AD = re.split(r'GCTAGC', read)[1]
    except:
        if read[:3] == "ACT" or read[:3]=="GCT":
            AD = read[27:]
        elif read[:3] == "TAC":
            AD = read[28:]
        elif read[:3] == "CTA":
            AD = read[29:]
        else: AD = "notfound"
    return(AD)

def seqCounter(fastq):
    seqCounts = {}
    for line in seqlib.read_fastq(fastq):
        AD = ADfinder(line[1])[:120]
        if AD not in seqCounts: seqCounts[AD] = 1
        else: seqCounts[AD] += 1
    df = pd.DataFrame.from_dict(seqCounts, orient="index", columns=["numreads"])
    df.reset_index(inplace=True)
    df.rename(columns={"index":"ArrayDNA"}, inplace=True)
    return df

def perfect_matcher(fastq, design, write_to_name = None):
    sample = seqCounter(fastq)
    design = design[["ArrayDNA"]]
    t = design.merge(sample, on = "ArrayDNA").sort_values(by = "numreads")
    print("\nFile:",fastq,"total reads:",sample["numreads"].sum(),
          "\nperfect matches:", t["numreads"].sum(),
          "\nperfect matches/total reads:", t["numreads"].sum()/sample["numreads"].sum(),
          "\ngarbage reads:", sample.where(sample["ArrayDNA"] =="notfound")["numreads"].sum(),
          "\nunmatched reads:", sample["numreads"].sum() - t["numreads"].sum())
    t=t[["numreads","ArrayDNA"]]
        # final output df
    if write_to_name != None:
        # max input: num, seq (no header)
        t.to_csv(f"{write_to_name}.csv", index=False, header=False)
    return t 

In [4]:
# bacterial libraries
repOne = perfect_matcher("../Data/Bacterial_Libraries/LC_M1_checkstep1_rep1.fastq", design, write_to_name="../Data/Bacterial_Libraries/LC_M1_bacterial_library_rep1")
repTwo = perfect_matcher("../Data/Bacterial_Libraries/LC_M1_checkstep1_rep2.fastq", design, write_to_name="../Data/Bacterial_Libraries/LC_M1_bacterial_library_rep2")


File: ../Data/Bacterial_Libraries/LC_M1_checkstep1_rep1.fastq total reads: 1443210 
perfect matches: 1124160 
perfect matches/total reads: 0.7789303012035671 
garbage reads: 17791.0 
unmatched reads: 319050

File: ../Data/Bacterial_Libraries/LC_M1_checkstep1_rep2.fastq total reads: 1014344 
perfect matches: 793373 
perfect matches/total reads: 0.7821537860922921 
garbage reads: 9835.0 
unmatched reads: 220971


In [ ]:
# ratio sort - fill in file names or use a loop
bin1 = perfect_matcher("../Data/SC_glucose_Ratio_8bins/-", design, write_to_name="-")
bin2 = perfect_matcher("../Data/SC_glucose_Ratio_8bins/-", design, write_to_name="-")
bin3 = perfect_matcher("../Data/SC_glucose_Ratio_8bins/-", design, write_to_name="-")
bin4 = perfect_matcher("../Data/SC_glucose_Ratio_8bins/-", design, write_to_name="-")
bin5 = perfect_matcher("../Data/SC_glucose_Ratio_8bins/-", design, write_to_name="-")
bin6 = perfect_matcher("../Data/SC_glucose_Ratio_8bins/-", design, write_to_name="-")
bin7 = perfect_matcher("../Data/SC_glucose_Ratio_8bins/-", design, write_to_name="-")
bin8 = perfect_matcher("../Data/SC_glucose_Ratio_8bins/-", design, write_to_name="-")